In [17]:
text = ("Trust me, though, the words were on their way, and when "
        "they arrived, Liesel would hold them in her hands like "
        "the clouds, and she would wring them out, like the rain.")
tokens = text.split()
tokens[:8]

['Trust', 'me,', 'though,', 'the', 'words', 'were', 'on', 'their']

In [18]:
import re
pattern = r'\w+(?:\'\w+)?|[^\w\s]' #(?:\'\w+)? detects whether the word contains a single apostrophe followed by one or more letters
    #can accommodate double contractions with r'\w+(?:\'\w+){0,2}|[^\w\s]'
texts = [text]
texts.append("There's no such thing as survival of the fittest. Survival of the most adequate, maybe.")
tokens = list(re.findall(pattern, texts[-1]))
tokens[:8]

["There's", 'no', 'such', 'thing', 'as', 'survival', 'of', 'the']

In [22]:
tokens[8:16]

['=', '"', 'en', '"', '>', '<', 'head', '>']

In [26]:
tokens[16:]

['maybe', '.']

In [19]:
import numpy as np
vocab = sorted(set(tokens)) #coerce the list into a set so that vocabulary contains only unique tokens
' '.join(vocab[:12]) #sorts lexicographically so the order is punctuation, capitalized, lowercase

", . Survival There's adequate as fittest maybe most no of such"

In [28]:
num_tokens = len(tokens)
num_tokens

18

In [20]:
vocab_size = len(vocab)
vocab_size

15

In [21]:
import spacy
#spacy.cli.download('en_core_web_sm') #运行出错，显示404，手动下载文件放入site-packages
nlp = spacy.load('en_core_web_sm') #sm stands for small, md=medium, lg=large
doc = nlp(texts[-1])
type(doc)

AttributeError: partially initialized module 'torch' has no attribute 'fx' (most likely due to a circular import)

In [ ]:
tokens = [tok.text for tok in doc]
tokens[:9]

In [ ]:
tokens[9:17]

In [33]:
from spacy import displacy
sentence = list(doc.sents)[0]
svg = displacy.render(sentence, style="dep", jupyter=False)
open('sentence_diagram.svg', 'w').write(svg)
# displacy.serve(sentence, style="dep")
# !firefox 127.0.0.1:5000
displacy.render(sentence, style="dep")

In [34]:
import requests
text = requests.get('https://proai.org/nlpia2-ch2.adoc').text #ConnectTimeout
#text = requests.get('https://cgtn.com').text
f'{round(len(text) / 10_1000)}0k'

'20k'

In [40]:
%timeit nlp(text)

6.01 s ± 146 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [36]:
f'{round(len(text) / 10_000)}0k'

'190k'

In [37]:
doc = nlp(text)
f'{round(len(list(doc)) / 10_000)}0k'

'40k'

In [38]:
f'{round(len(doc) / 1_000 / 10.4)}kWPS' #kWPS is for thousands of words per second

'4kWPS'

In [39]:
nlp.pipe_names

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

In [41]:
nlp = spacy.load('en_core_web_sm', disable=nlp.pipe_names) #disable spaCy pipeline to get faster
%timeit nlp(text)

439 ms ± 4.65 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [42]:
import nltk
#nltk.download('punkt')
from nltk.tokenize import word_tokenize
%timeit word_tokenize(text)

286 ms ± 15.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [43]:
tokens = word_tokenize(text)
f'{round(len(tokens) / 10_000)}0k'

'40k'

In [44]:
pattern = r'\w+(?:\'\w+)?|[^\w\s]'
tokens = re.findall(pattern, text)
f'{round(len(tokens) / 10_000)}0k'

'40k'

In [45]:
%timeit re.findall(pattern, text)

16.1 ms ± 1.01 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [46]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(ngram_range=(1, 2), analyzer='char')
vectorizer.fit(texts)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'char'


In [47]:
bpevocab_list = [sorted((i, s) for s, i in vectorizer.vocabulary_.items())]
bpevocab_dict = dict(bpevocab_list[0])
list(bpevocab_dict.values())[:7]

[' ', ' a', ' c', ' f', ' h', ' i', ' l']

In [48]:
vectors = vectorizer.transform(texts)
df = pd.DataFrame(vectors.todense(), columns=vectorizer.get_feature_names_out())
df.index = [t[:8] + '...' for t in texts]
df = df.T
df['total'] = df.T.sum()
df

,Trust me...,There's ...,total
,31,14,45
a,3,2,5
c,1,0,1
f,0,1,1
h,3,0,3
...,...,...,...
wr,1,0,1
y,2,1,3
y,1,0,1
"y,",1,0,1


In [49]:
df.sort_values('total').tail()

,Trust me...,There's ...,total
he,10,3,13
h,14,5,19
t,11,9,20
e,18,8,26
,31,14,45


In [50]:
df['n'] = [len(tok) for tok in vectorizer.vocabulary_]
df[df['n'] > 1].sort_values('total').tail()

,Trust me...,There's ...,total,n
th,8,4,12,2
he,10,3,13,2
h,14,5,19,2
t,11,9,20,2
e,18,8,26,2


In [52]:
import requests
url = ("https://gitlab.com/tangibleai/nlpia/-/raw/master/src/nlpia/data/stopword_lists.json")
response = requests.get(url)
stopwords = response.json()['exhaustive']
tokens = 'the words were just as I remembered them'.split()
tokens_without_stopwords = [x for x in tokens if x not in stopwords]
print(tokens_without_stopwords)

['I', 'remembered']


In [54]:
import nltk
nltk.download('stopwords')
stop_words = nltk.corpus.stopwords.words('english')
len(stop_words)

[nltk_data] Downloading package stopwords to C:\Users\throw\Study\Mini
[nltk_data]     conda\miniconda3\envs\nlpia2\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


198

In [55]:
stop_words[:7]

['a', 'about', 'above', 'after', 'again', 'against', 'ain']

In [56]:
[sw for sw in stopwords if len(sw) == 1]

['a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [57]:
tokens = ['House', 'Visitor', 'Center']
normalized_tokens = [x.lower() for x in tokens]
print(normalized_tokens)

['house', 'visitor', 'center']


In [59]:
def stem(phrase):
    return ' '.join([re.findall('^(.*ss|.*?)(s)?$',
                                word)[0][0].strip("'") for word in phrase.lower().split()])
stem('houses')

'house'

In [60]:
stem("Doctor House's calls")

'doctor house call'

In [61]:
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()
' '.join([stemmer.stem(w).strip("'") for w in "dish washer's fairly washed dishes".split()])

'dish washer fairli wash dish'

In [62]:
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer(language='english')
' '.join([stemmer.stem(w).strip("'") for w in "dish washer's fairly washed dishes".split()])

'dish washer fair wash dish'

In [66]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to C:\Users\throw\Study\Minico
[nltk_data]     nda\miniconda3\envs\nlpia2\nltk_data...


True

In [64]:
nltk.download('omw-1.4')

[nltk_data] Downloading package omw-1.4 to C:\Users\throw\Study\Minico
[nltk_data]     nda\miniconda3\envs\nlpia2\nltk_data...


True

In [70]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
lemmatizer.lemmatize("better")

'better'

In [71]:
lemmatizer.lemmatize("better", pos="a")

'good'

In [72]:
lemmatizer.lemmatize("good", pos="a")

'good'

In [73]:
lemmatizer.lemmatize("goods", pos="a")

'goods'

In [74]:
lemmatizer.lemmatize("goods", pos="n")

'good'

In [75]:
lemmatizer.lemmatize("goodness", pos="n")

'goodness'

In [76]:
lemmatizer.lemmatize("best", pos="a")

'best'

In [67]:
stemmer.stem('goodness')

'good'

In [68]:
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp("better good goods goodness best")
for token in doc:
    print(token.text, token.lemma_)

better well
good good
goods good
goodness goodness
best good


In [1]:
import jieba
seg_list = jieba.cut("西安是一座举世闻名的文化古城")
list(seg_list)

C:\Users\throw\Study\Miniconda\miniconda3\envs\nlpia2\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\throw\AppData\Local\Temp\jieba.cache
Loading model cost 1.077 seconds.
Prefix dict has been built successfully.


['西安', '是', '一座', '举世闻名', '的', '文化', '古城']

In [2]:
#accurate mode
seg_list = jieba.cut("西安是一座举世闻名的文化古城", cut_all=True)
list(seg_list)

['西安', '是', '一座', '举世', '举世闻名', '闻名', '的', '文化', '古城']

In [22]:
import pandas as pd
onehot_vectors = np.zeros((len(tokens), vocab_size), int)
for i, tok in enumerate(tokens):
    if tok not in vocab:
        continue
    onehot_vectors[i, vocab.index(tok)] = 1
df_onehot = pd.DataFrame(onehot_vectors, columns=vocab)
df_onehot.shape

(18, 15)

In [23]:
df_onehot.iloc[:,:8].replace(0, '')

,",",.,Survival,There's,adequate,as,fittest,maybe
0,,,,1,,,,
1,,,,,,,,
2,,,,,,,,
3,,,,,,,,
4,,,,,,1,,
5,,,,,,,,
6,,,,,,,,
7,,,,,,,,
8,,,,,,,1,
9,,1,,,,,,


In [24]:
bow = sorted(set(re.findall(pattern, text)))
bow[:9]

[',', '.', 'Liesel', 'Trust', 'and', 'arrived', 'clouds', 'hands', 'her']

In [26]:
bow[9:19]

['hold', 'in', 'like', 'me', 'on', 'out', 'rain', 'she', 'the', 'their']

In [27]:
bow[19:27]

['them', 'they', 'though', 'way', 'were', 'when', 'words', 'would']

In [28]:
pip install vaderSentiment

Note: you may need to restart the kernel to use updated packages.


In [1]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sa = SentimentIntensityAnalyzer()
sa.lexicon

{'$:': -1.5,
 '%)': -0.4,
 '%-)': -1.5,
 '&-:': -0.4,
 '&:': -0.7,
 "( '}{' )": 1.6,
 '(%': -0.9,
 "('-:": 2.2,
 "(':": 2.3,
 '((-:': 2.1,
 '(*': 1.1,
 '(-%': -0.7,
 '(-*': 1.3,
 '(-:': 1.6,
 '(-:0': 2.8,
 '(-:<': -0.4,
 '(-:o': 1.5,
 '(-:O': 1.5,
 '(-:{': -0.1,
 '(-:|>*': 1.9,
 '(-;': 1.3,
 '(-;|': 2.1,
 '(8': 2.6,
 '(:': 2.2,
 '(:0': 2.4,
 '(:<': -0.2,
 '(:o': 2.5,
 '(:O': 2.5,
 '(;': 1.1,
 '(;<': 0.3,
 '(=': 2.2,
 '(?:': 2.1,
 '(^:': 1.5,
 '(^;': 1.5,
 '(^;0': 2.0,
 '(^;o': 1.9,
 '(o:': 1.6,
 ")':": -2.0,
 ")-':": -2.1,
 ')-:': -2.1,
 ')-:<': -2.2,
 ')-:{': -2.1,
 '):': -1.8,
 '):<': -1.9,
 '):{': -2.3,
 ');<': -2.6,
 '*)': 0.6,
 '*-)': 0.3,
 '*-:': 2.1,
 '*-;': 2.4,
 '*:': 1.9,
 '*<|:-)': 1.6,
 '*\\0/*': 2.3,
 '*^:': 1.6,
 ',-:': 1.2,
 "---'-;-{@": 2.3,
 '--<--<@': 2.2,
 '.-:': -1.2,
 '..###-:': -1.7,
 '..###:': -1.9,
 '/-:': -1.3,
 '/:': -1.3,
 '/:<': -1.4,
 '/=': -0.9,
 '/^:': -1.0,
 '/o:': -1.4,
 '0-8': 0.1,
 '0-|': -1.2,
 '0:)': 1.9,
 '0:-)': 1.4,
 '0:-3': 1.5,
 '0:03': 1.9,
 '

In [3]:
[(tok, score) for tok, score in sa.lexicon.items() if " " in tok]

[("( '}{' )", 1.6),
 ("can't stand", -2.0),
 ('fed up', -1.8),
 ('screwed up', -1.5)]

In [4]:
sa.polarity_scores(text="Python is very readable and it's great for NLP.")

{'neg': 0.0, 'neu': 0.661, 'pos': 0.339, 'compound': 0.6249}

In [5]:
sa.polarity_scores(text="Python is not a bad choice for most applications.")

{'neg': 0.0, 'neu': 0.737, 'pos': 0.263, 'compound': 0.431}

In [7]:
corpus = ["Absolutely perfect! Love it! :-) :-) :-)", "Horrible! Completely useless. :(",
          "It was OK. Some good and some bad things."]
for doc in corpus:
    scores = sa.polarity_scores(doc)
    print('{:+}: {}'.format(scores['compound'], doc))

+0.9428: Absolutely perfect! Love it! :-) :-) :-)
-0.8768: Horrible! Completely useless. :(
-0.1531: It was OK. Some good and some bad things.


In [9]:
import pandas as pd
movies = pd.read_csv('https://proai.org/movie-reviews.csv.gz', index_col=0)
movies.head().round(2)

,sentiment,text
id,,
1,2.27,The Rock is destined to be the 21st Century's ...
2,3.53,The gorgeously elaborate continuation of ''The...
3,-0.60,Effective but too tepid biopic
4,1.47,If you sometimes like to go to the movies to h...
5,1.73,"Emerges as something rare, an issue movie that..."


In [10]:
movies.describe().round(2)

,sentiment
count,10605.00
mean,0.00
std,1.92
min,-3.88
25%,-1.77
50%,-0.08
75%,1.83
max,3.94


In [11]:
pd.options.display.width = 75
from nltk.tokenize import casual_tokenize
bows = []
from collections import Counter
for text in movies.text:
    bows.append(Counter(casual_tokenize(text)))
df_movies = pd.DataFrame.from_records(bows)
df_movies = df_movies.fillna(0).astype(int)
df_movies.shape

(10605, 20756)

In [15]:
df_movies.head()

,The,Rock,is,destined,to,be,the,21st,Century's,new,...,Ill,slummer,Rashomon,dipsticks,Bearable,Staggeringly,’,ve,muttering,dissing
0,1,1,1,1,2,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,1,0,4,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
df_movies.head()[list(bows[0].keys())]

,The,Rock,is,destined,to,be,the,21st,Century's,new,...,Schwarzenegger,",",Jean,Claud,Van,Damme,or,Steven,Segal,.
0,1,1,1,1,2,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
1,2,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,4
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,1,0,4,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1


In [21]:
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB()
nb = nb.fit(df_movies, movies.sentiment > 0)
movies['pred_senti'] = (
    nb.predict_proba(df_movies))[:, 1] * 8 - 4
movies['error'] = movies.pred_senti - movies.sentiment
mae = movies['error'].abs().mean().round(1) #Mean Absolute Error
mae

1.9

In [23]:
movies['senti_ispos'] = (movies['sentiment'] > 0).astype(int)
movies['pred_ispos'] = (movies['pred_senti'] > 0).astype(int)
columns = [c for c in movies.columns if 'senti' in c or 'pred' in c]
movies[columns].head(8)

,sentiment,pred_senti,senti_ispos,pred_ispos
id,,,,
1,2.266667,2.511515,1,1
2,3.533333,3.999904,1,1
3,-0.600000,-3.655976,0,0
4,1.466667,1.940954,1,1
5,1.733333,3.910373,1,1
6,2.533333,3.995188,1,1
7,2.466667,3.960466,1,1
8,1.266667,-1.918701,1,0


In [19]:
(movies.pred_ispos == movies.senti_ispos).sum() / len(movies)

0.9344648750589345

In [24]:
products = pd.read_csv('https://proai.org/product-reviews.csv.gz')
products.columns

Index(['id', 'sentiment', 'text'], dtype='object')

In [25]:
products.head()

,id,sentiment,text
0,1_1,-0.90,troubleshooting ad-2500 and ad-2600 no picture...
1,1_2,-0.15,"repost from january 13, 2004 with a better fit..."
2,1_3,-0.20,does your apex dvd player only play dvd audio ...
3,1_4,-0.10,or does it play audio and video but scrolling ...
4,1_5,-0.50,before you try to return the player or waste h...


In [26]:
bows = []
for text in products['text']:
    bows.append(Counter(casual_tokenize(text)))
df_products = pd.DataFrame.from_records(bows)
df_products = df_products.fillna(0).astype(int)
df_products.shape

(3546, 5687)

In [27]:
df_all_bows = pd.concat([df_movies, df_products])
df_all_bows.columns

Index(['The', 'Rock', 'is', 'destined', 'to', 'be', 'the', '21st',
       'Century's', 'new',
       ...
       'sligtly', 'owner', '81', 'defectively', 'warrranty', 'expire',
       'expired', 'voids', 'baghdad', 'harddisk'],
      dtype='object', length=23302)

In [30]:
vocab = list(df_movies.columns)
df_products = df_all_bows.iloc[len(movies):]
df_products = df_products[vocab]
df_products = df_products.fillna(0).astype(int)
df_products.shape

(3546, 20756)

In [29]:
df_movies.shape

(10605, 20756)

In [31]:
products['senti_ispos'] = (products['sentiment'] > 0).astype(int)
products['pred_ispos'] = nb.predict(df_products).astype(int)
correct = (products['pred_ispos'] == products['senti_ispos'])
correct.sum() / len(products)

0.5572476029328821